In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
docs = text_splitter.split_documents(documents)

```markdown
1. Belgeleri Yükleme ve Parçalama
Amaç: speech.txt dosyasını yükleyip, 1000 karakterlik ve 30 karakter üst üste binecek şekilde parçalara ayırmak.
Kütüphaneler:
TextLoader: Düz metin dosyasını belgeye çevirir.
CharacterTextSplitter: Metni parçalara böler.
```

In [4]:
embeddings = OllamaEmbeddings(model="gemma:2b")
db = FAISS.from_documents(docs, embeddings)
db

```markdown
2. Embedding Modeli ve FAISS Vektör Veritabanı Oluşturma
Amaç: Belgeleri embedding vektörlerine dönüştürüp FAISS vektör veritabanında saklamak.
Kütüphaneler:
OllamaEmbeddings: Metinleri vektörlere dönüştürür.
FAISS: Hızlı benzerlik araması için vektör veritabanı sağlar.
```

In [5]:
query = "What is the main topic of the speech?"
results = db.similarity_search(query, k=3)
results

[Document(id='bc09bbf9-d1ef-47b9-a4fb-ba4219e45bd6', metadata={'source': 'speech.txt'}, page_content='When I addressed the Congress on the 26th of February last, I thought that it would suffice to assert our neutral rights with arms, our right to use the seas against unlawful interference, our right to keep our people safe against unlawful violence. But armed neutrality, it now appears, is impracticable... Armed neutrality is ineffectual enough at best; in such circumstances and in the face of such pretensions it is worse than ineffectual: it is likely only to produce what it was meant to prevent; it is practically certain to draw us into the war without either the rights or the effectiveness of belligerents. There is one choice we cannot make, we are incapable of making: we will not choose the path of submission and suffer the most sacred rights of our nation and our people to be ignored or violated. The wrongs against which we now array ourselves are no common wrongs; they cut to the

```markdown
3. Sorgu ile Benzer Belgeleri Arama
Amaç: Sorguya en yakın 3 belgeyi bulmak.

```

In [6]:
retriver = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retriver.invoke(query)

[Document(id='bc09bbf9-d1ef-47b9-a4fb-ba4219e45bd6', metadata={'source': 'speech.txt'}, page_content='When I addressed the Congress on the 26th of February last, I thought that it would suffice to assert our neutral rights with arms, our right to use the seas against unlawful interference, our right to keep our people safe against unlawful violence. But armed neutrality, it now appears, is impracticable... Armed neutrality is ineffectual enough at best; in such circumstances and in the face of such pretensions it is worse than ineffectual: it is likely only to produce what it was meant to prevent; it is practically certain to draw us into the war without either the rights or the effectiveness of belligerents. There is one choice we cannot make, we are incapable of making: we will not choose the path of submission and suffer the most sacred rights of our nation and our people to be ignored or violated. The wrongs against which we now array ourselves are no common wrongs; they cut to the

```markdown
4. Retriever ile Sorgu
Amaç: Retriever arayüzüyle benzer belgeleri almak.

```

In [8]:
# Similarity search with score
results_with_scores = db.similarity_search_with_score(query, k=3)
results_with_scores
# Displaying results with scores
for doc, score in results_with_scores:
    print(f"Score: {score}\nDocument: {doc.page_content}\n")


Score: 3176.18994140625
Document: When I addressed the Congress on the 26th of February last, I thought that it would suffice to assert our neutral rights with arms, our right to use the seas against unlawful interference, our right to keep our people safe against unlawful violence. But armed neutrality, it now appears, is impracticable... Armed neutrality is ineffectual enough at best; in such circumstances and in the face of such pretensions it is worse than ineffectual: it is likely only to produce what it was meant to prevent; it is practically certain to draw us into the war without either the rights or the effectiveness of belligerents. There is one choice we cannot make, we are incapable of making: we will not choose the path of submission and suffer the most sacred rights of our nation and our people to be ignored or violated. The wrongs against which we now array ourselves are no common wrongs; they cut to the very roots of human life.

Score: 3221.515625
Document: I have call

```markdown
5. Skorlarla Benzerlik Araması
Amaç: Benzer belgeleri ve benzerlik skorlarını birlikte görmek.

```

In [9]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

[-0.2216968834400177,
 -1.4594179391860962,
 0.04175268113613129,
 1.3306292295455933,
 2.2590956687927246,
 2.094722270965576,
 1.5485420227050781,
 0.013651345856487751,
 0.14809277653694153,
 -0.5808722376823425,
 2.0421340465545654,
 0.9328432679176331,
 0.8309697508811951,
 1.6767103672027588,
 -0.5568175315856934,
 -0.48392319679260254,
 2.4952609539031982,
 1.6788465976715088,
 -0.4355024993419647,
 -0.10034627467393875,
 2.640976905822754,
 -0.387329638004303,
 0.25670549273490906,
 -1.1861199140548706,
 0.03440987691283226,
 -1.5759005546569824,
 -0.25240468978881836,
 0.8928446173667908,
 1.0484150648117065,
 -0.5874263048171997,
 -0.15426136553287506,
 0.25974780321121216,
 -0.44210153818130493,
 -0.4141502380371094,
 0.29593807458877563,
 0.7072330713272095,
 0.32057029008865356,
 0.5346298217773438,
 0.6970640420913696,
 -0.5594256520271301,
 -2.077662467956543,
 -0.9791064262390137,
 0.6519126892089844,
 -0.9759610295295715,
 -0.29215386509895325,
 -1.0008976459503174,
 1

```markdown
6. Sorgunun Embedding Vektörünü Alma
Amaç: Sorgu cümlesinin embedding vektörünü elde etmek.

```

In [11]:
docs_score = db.similarity_search_by_vector(embedding_vector, k=3)
docs_score


[Document(id='bc09bbf9-d1ef-47b9-a4fb-ba4219e45bd6', metadata={'source': 'speech.txt'}, page_content='When I addressed the Congress on the 26th of February last, I thought that it would suffice to assert our neutral rights with arms, our right to use the seas against unlawful interference, our right to keep our people safe against unlawful violence. But armed neutrality, it now appears, is impracticable... Armed neutrality is ineffectual enough at best; in such circumstances and in the face of such pretensions it is worse than ineffectual: it is likely only to produce what it was meant to prevent; it is practically certain to draw us into the war without either the rights or the effectiveness of belligerents. There is one choice we cannot make, we are incapable of making: we will not choose the path of submission and suffer the most sacred rights of our nation and our people to be ignored or violated. The wrongs against which we now array ourselves are no common wrongs; they cut to the

```markdown
7. Vektörle Benzerlik Arama
Amaç: Vektörle doğrudan benzerlik araması yapmak.

```

In [13]:
# Save the vector store to disk
db.save_local("faiss_index")


```markdown
8. Vektör Veritabanını Kaydetme
Amaç: FAISS vektör veritabanını diske kaydetmek.

```

In [15]:
# Load the vector store from disk
db_loaded = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
docs = db_loaded.similarity_search(query, k=3)
docs
# Displaying loaded results
for doc in docs:
    print(f"Document: {doc.page_content}\n")

    

Document: When I addressed the Congress on the 26th of February last, I thought that it would suffice to assert our neutral rights with arms, our right to use the seas against unlawful interference, our right to keep our people safe against unlawful violence. But armed neutrality, it now appears, is impracticable... Armed neutrality is ineffectual enough at best; in such circumstances and in the face of such pretensions it is worse than ineffectual: it is likely only to produce what it was meant to prevent; it is practically certain to draw us into the war without either the rights or the effectiveness of belligerents. There is one choice we cannot make, we are incapable of making: we will not choose the path of submission and suffer the most sacred rights of our nation and our people to be ignored or violated. The wrongs against which we now array ourselves are no common wrongs; they cut to the very roots of human life.

Document: I have called the Congress into extraordinary session 

```markdown
9. Kaydedilen Veritabanını Yükleme ve Kullanma
Amaç: Kaydedilmiş FAISS veritabanını tekrar yükleyip sorgu yapmak.

```

```markdown
Genel Amaç
Bu notebook'ta amaç, metin belgelerini embedding vektörlerine dönüştürüp FAISS ile hızlı benzerlik araması yapmaktır. Belgeler parçalara ayrılır, embedding ile vektörleştirilir, FAISS veritabanında saklanır ve sorgularla benzer belgeler bulunabilir. Ayrıca veritabanı kaydedilip tekrar yüklenebilir.
```

```markdown
FAISS ve Chroma ikisi de vektör veritabanı (vector store) olarak kullanılır ve metin embedding’leriyle benzerlik araması yapmayı sağlar. Ancak bazı önemli farkları ve benzerlikleri vardır:

Benzerlikler
Amaç: Her ikisi de metinleri embedding vektörlerine dönüştürüp, bu vektörler üzerinde hızlı benzerlik (similarity) araması yapar.
Kullanım: LangChain, LlamaIndex gibi framework’lerle kolayca entegre olur.
API: Her ikisi de .from_documents, .similarity_search, .save_local gibi benzer fonksiyonlara sahiptir.
Destek: Hem OpenAI, Ollama, HuggingFace gibi farklı embedding modelleriyle çalışabilirler.
Farklılıklar
Özellik	FAISS	Chroma
Altyapı	Facebook tarafından geliştirilen C++ tabanlı, Python binding’li bir kütüphane	Python tabanlı, modern, açık kaynak bir vektör veritabanı
Kurulum	Ekstra bağımlılık ve bazen derleme gerekebilir	Sadece Python paketi olarak kolay kurulur
Performans	Çok büyük veri setlerinde yüksek performans, GPU desteği	Büyük veri setlerinde iyi performans, ancak FAISS kadar optimize değil
Özellikler	Sadece vektör arama (vektör+metadata desteği sınırlı)	Vektör arama + metadata + filtreleme + kalıcı depolama (persist)
Kalıcı Depolama	Disk’e kaydetme ve yükleme var, ancak daha temel	Kalıcı depolama, segmentasyon, koleksiyon yönetimi daha gelişmiş
Filtreleme	Sınırlı metadata filtreleme	Gelişmiş metadata filtreleme ve sorgulama
Topluluk	Daha eski ve yaygın, araştırma odaklı	Yeni, hızlı gelişen, uygulama odaklı
Kısaca
FAISS: Çok büyük veri setlerinde, yüksek hız ve ölçeklenebilirlik isteyen projelerde tercih edilir. Daha teknik ve düşük seviyeli.
Chroma: Modern Python projelerinde, kolay kurulum, gelişmiş metadata yönetimi ve kalıcı depolama isteyenler için uygundur.
Her ikisi de LangChain ile kolayca kullanılabilir; seçim, projenin ihtiyaçlarına ve altyapısına göre yapılır.
```